<a href="https://colab.research.google.com/github/subiksha0515/Recommendation_of_RAG/blob/main/Final_Adaptive_Rag_3_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Title : Adaptive RAG with  Hallucination Control
---

What is Adaptive RAG?
---

Adaptive Retrieval-Augmented Generation (Adaptive RAG) is an intelligent RAG pipeline where the system:

- Retrieves relevant chunks from a document using embeddings + FAISS

- Generates answers strictly from retrieved context

- Detects hallucination using similarity check

- Automatically rewrites the query if hallucination occurs

- Falls back to web knowledge if the document does not contain the answer

It is called Adaptive because the system changes its behavior based on the situation:

- If related → use document

- If not related → web fallback

- If hallucinated → rewrite query and retry

---

🎯 Objective of the Project
---

- Convert a document into semantic chunks

- Represent chunks in vector (embedding) space

- Visualize chunk relationships in 3D semantic space

- Retrieve only relevant content for a user query

- Prevent hallucinated answers

- Build a self-correcting RAG system

---

🧩 Stage-wise Models & Techniques Used
---
| Stage                   | Purpose                      | Model / Technique Used   |
| ----------------------- | ---------------------------- | ------------------------ |
| Semantic Chunking       | Split document meaningfully  | NLTK sentence tokenizer  |
| Embedding               | Convert text to vectors      | `text-embedding-3-small` |
| Vector Storage          | Store embeddings             | FAISS IndexFlatL2        |
| Retrieval               | Find nearest chunks          | Vector similarity search |
| Answer Generation       | Generate answer from context | `gpt-4.1-nano`           |
| Hallucination Detection | Verify answer vs chunks      | Cosine similarity        |
| Query Rewriting         | Improve failed query         | Prompt engineering       |
| Web Fallback            | Answer if not in document    | LLM web-style response   |
| Visualization           | Show chunk relationships     | PCA + Plotly 3D          |


🏗️ System Architecture
---
<pre>

                ┌───────────────────────────┐
                │       Document (.txt)     │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │   Semantic Chunking      │
                │          (NLTK)          │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │  OpenAI Embedding Model  │
                │   (text-embedding-3)     │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │     FAISS Vector Index   │
                └──────────────┬────────────┘
                               ↓
                               ↓
                ┌───────────────────────────┐
                │        User Query        │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │      Query Embedding     │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │ Similarity Check (FAISS) │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │ Retrieve Top Chunks      │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │  GPT Answer from Context │
                └──────────────┬────────────┘
                               ↓
                ┌───────────────────────────┐
                │   Hallucination Check    │
                └──────────────┬────────────┘
                      ┌────────┴────────┐
                      ↓                 ↓
           ┌──────────────────┐   ┌──────────────────┐
           │   Verified       │   │   Rewrite Query  │
           │   Final Answer   │   └─────────┬────────┘
           └──────────────────┘             ↓
                                   ┌──────────────────┐
                                   │   Retry Process  │
                                   └─────────┬────────┘
                                             ↓
                                   If not related → Web Fallback
```
</pre>


---

📊 Purpose of the 3D Visualization
---

Each dot = one document chunk

Distance between dots = similarity in meaning

This is the same semantic space FAISS uses to retrieve answers

Helps visually understand how RAG “sees” the document

---

In [ ]:
!pip install --upgrade pip -q
!pip install gradio==5.23.3 plotly numpy requests nltk faiss-cpu scikit-learn openai -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.9 MB/s eta 0:00:00


CELL 1: Install Dependencies


In [ ]:
!pip install --upgrade pip -q


In [ ]:
!pip install numpy requests nltk -q
!pip install faiss-cpu scikit-learn -q
!pip install openai gradio plotly -q


CELL 2: Import Libraries & Download Tokenizers

In [ ]:
import numpy as np
import faiss
import nltk
from nltk.tokenize import sent_tokenize
from openai import OpenAI
from google.colab import userdata
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

CELL 3: Load API Credentials from Colab

In [ ]:
API_KEY = userdata.get("API_KEY")
BASE_URL = userdata.get("BASE_URL")

if not API_KEY or not BASE_URL:
    raise RuntimeError("❌ API_KEY / BASE_URL missing in Colab Secrets")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("✅ API loaded successfully")


✅ API loaded successfully


CELL 4: Load External Document

In [ ]:
with open("/content/al.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("📄 Document length:", len(raw_text))


📄 Document length: 908


CELL 5: Semantic Chunking

In [ ]:
def semantic_chunking(text, max_sentences=5):
    sentences = sent_tokenize(text)
    chunks = []

    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences])
        chunks.append(chunk)

    return chunks

documents = semantic_chunking(raw_text)
print("🧩 Total semantic chunks:", len(documents))


🧩 Total semantic chunks: 2


CELL 6: API-Based Embeddings

In [ ]:
def embed_texts(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([item.embedding for item in response.data], dtype="float32")

doc_embeddings = embed_texts(documents)
print("📐 Embedding shape:", doc_embeddings.shape)


📐 Embedding shape: (2, 1536)


In [ ]:
def embed_query(query):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    return np.array(response.data[0].embedding, dtype="float32").reshape(1, -1)


CELL 7:Build FAISS Index

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

print("📦 Indexed chunks:", index.ntotal)


📦 Indexed chunks: 2


CELL 8:Build Vector Database

In [ ]:
def is_related_to_index(query_embedding, threshold=1.5):
    distances, _ = index.search(query_embedding, 1)
    return distances[0][0] < threshold


def retrieve(query_embedding, k=3):
    _, idx = index.search(query_embedding, k)
    return [documents[i] for i in idx[0]]

CELL 9: Answer Generation

In [ ]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer ONLY using the provided context. "
                    "If the answer is not explicitly present, reply exactly: "
                    "'NOT FOUND IN CONTEXT'."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.5
    )
    return response.choices[0].message.content


CELL 10:Hallucination Detection

In [ ]:
def hallucinated(answer, chunks, threshold=0.3):
    texts = chunks + [answer]
    embeddings = embed_texts(texts)

    answer_emb = embeddings[-1]
    chunk_embs = embeddings[:-1]

    similarities = [
        np.dot(answer_emb, c) / (np.linalg.norm(answer_emb) * np.linalg.norm(c))
        for c in chunk_embs
    ]

    max_similarity = max(similarities)
    print(f"  Generated Answer: '{answer}'")
    print(f"  Max similarity to chunks: {max_similarity:.4f} (Threshold: {threshold})")

    return max_similarity < threshold

CELL 11: Query Processing

In [ ]:
def preprocess_query(query):
    if len(query.split()) < 3 and not query.lower().startswith("what is"):
        return f"What is {query}?"
    return query


def rewrite_query(query):
    return f"Explain clearly with examples: {query}"


CELL 12:Web Search

In [ ]:
def web_search(query):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": "You are a web-enabled assistant. Provide factual answers."
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


CELL 13: Keyword_overlap

In [ ]:
def keyword_overlap(query, documents, min_hits=1):
    query_terms = set(query.lower().split())
    for doc in documents:
        if len(query_terms & set(doc.lower().split())) >= min_hits:
            return True
    return False

CELL 14: Adaptive RAG Controller

In [ ]:
def adaptive_rag(query, max_loops=3):
    original_query = query

    for loop in range(1, max_loops + 1):
        print(f"\n🔁 LOOP {loop}: {query}")

        processed_query = preprocess_query(query)

        # 🔥 VECTORIZE QUERY
        query_emb = embed_query(processed_query)

        if not is_related_to_index(query_emb) or not keyword_overlap(processed_query, documents):
            print("❌ Not grounded in document → Web fallback")
            return web_search(processed_query)

        docs = retrieve(query_emb)
        context = " ".join(docs)

        prompt = f"""
Use ONLY the context below.

Context:
{context}

Question:
{query}

Answer:
"""

        answer = generate_answer(prompt)

        if answer.strip() == "NOT FOUND IN CONTEXT":
            return web_search(processed_query)

        if not hallucinated(answer, docs):
            print("✅ Answer verified")
            return answer

        print("⚠️ Hallucination detected → rewriting query")
        query = rewrite_query(original_query)

    return "⚠️ Unable to verify answer after multiple attempts."

CELL 15: Run the Adaptive RAG System

In [ ]:
result = adaptive_rag("what is ai")
print("\n🟢 FINAL ANSWER:\n", result)


🔁 LOOP 1: what is ai
  Generated Answer: 'Artificial Intelligence (AI) is a branch of computer science focused on building systems capable of performing tasks that require human intelligence.'
  Max similarity to chunks: 0.8368 (Threshold: 0.3)
✅ Answer verified

🟢 FINAL ANSWER:
 Artificial Intelligence (AI) is a branch of computer science focused on building systems capable of performing tasks that require human intelligence.


🟧 CELL 16: Import for UI & 3D View

In [ ]:
import gradio as gr
import plotly.express as px
from sklearn.decomposition import PCA


🟧 CELL 17: 3D Embedding Visualization Function

In [ ]:
def visualize_embeddings_3d(embeddings, texts):
    import numpy as np
    import plotly.express as px
    from sklearn.decomposition import PCA

    n_samples, n_features = embeddings.shape
    n_components = min(3, n_samples, n_features)

    pca = PCA(n_components=n_components)
    reduced = pca.fit_transform(embeddings)

    # Pad to 3D if needed
    if n_components < 3:
        pad = np.zeros((reduced.shape[0], 3 - n_components))
        reduced = np.hstack((reduced, pad))

    # Hover preview text
    hover_labels = []
    for i, chunk in enumerate(texts):
        preview = chunk.strip().split(".")[0][:80]
        hover_labels.append(f"Chunk {i+1}: {preview}...")

    fig = px.scatter_3d(
        x=reduced[:, 0],
        y=reduced[:, 1],
        z=reduced[:, 2],
        hover_name=hover_labels,
        title="🧠 3D Semantic Space of Document Chunks"
    )

    # Axis labels + tight layout
    fig.update_layout(
        scene=dict(
            xaxis_title="X → Meaning Pattern 1",
            yaxis_title="Y → Meaning Pattern 2",
            zaxis_title="Z → Meaning Pattern 3",
            aspectmode="cube",
            camera=dict(eye=dict(x=1.6, y=1.6, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        height=450
    )

    # 📝 Simple explanation shown on plot
    fig.add_annotation(
        text=(
            "Each blue dot = one document chunk<br>"
            "Close dots = similar topic<br>"
            "Far dots = different topic"
        ),
        showarrow=False,
        xref="paper", yref="paper",
        x=0.02, y=0.98,
        align="left",
        bordercolor="black",
        borderwidth=1,
        bgcolor="white",
        font=dict(size=12)
    )

    return fig


🟧 CELL 18: Pipeline Wrapper (connects everything)

In [ ]:
def run_full_rag(query: str) -> str:
    if not query:
        return "Please enter a question."

    print("🔍 Running Adaptive RAG...")
    answer = adaptive_rag(query)
    return answer


🟧 CELL 19: File Upload Handler (for Gradio)

In [ ]:
def load_new_document(file) -> tuple:
    global raw_text, documents, doc_embeddings, index

    # ✅ file is a path string from Gradio
    with open(file.name, "r", encoding="utf-8") as f:
        raw_text = f.read()

    documents = semantic_chunking(raw_text)
    doc_embeddings = embed_texts(documents)

    dimension = doc_embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(doc_embeddings)

    fig = visualize_embeddings_3d(doc_embeddings, documents)

    return (
        f"✅ Document Loaded\n🧩 Chunks: {len(documents)}\n📦 Indexed: {index.ntotal}",
        fig
    )


🟧 CELL 20: Gradio UI Design

In [ ]:
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="orange"),
    title="Adaptive RAG System"
) as demo:

    gr.Markdown("""
    # 🧠 Adaptive RAG System
    **Upload → Chunk → Embed → FAISS → Ask → Verified Answer**
    """)

    # ------------------ Upload Section ------------------
    with gr.Group():
        gr.Markdown("### 📄 Step 1: Upload Document")

        with gr.Row():
            file_input = gr.File(
                label="Upload .txt file",
                file_types=[".txt"]
            )
            status_output = gr.Textbox(
                label="System Status",
                interactive=False
            )

    # ------------------ 3D Visualization ------------------
    with gr.Group():
        gr.Markdown("### 📊 Step 2: Semantic Chunk Map")
        plot_output = gr.Plot(label="3D Meaning Space of Chunks")

    file_input.upload(
        fn=load_new_document,
        inputs=file_input,
        outputs=[status_output, plot_output]
    )

    # ------------------ Query Section ------------------
    with gr.Group():
        gr.Markdown("### ❓ Step 3: Ask Question From Document")

        query_input = gr.Textbox(
            label="Enter your question",
            placeholder="e.g. What is Artificial Intelligence?"
        )

        run_btn = gr.Button(
            "🚀 Run Adaptive RAG",
            variant="primary"
        )

        answer_output = gr.Textbox(
            label="Final Verified Answer",
            lines=6
        )

    run_btn.click(
        fn=run_full_rag,
        inputs=query_input,
        outputs=answer_output
    )

# Launch (stable for Colab)
demo.queue().launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3e34aa53dd8d811557.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🔍 Running Adaptive RAG...

🔁 LOOP 1: 5+6
❌ Not grounded in document → Web fallback
🔍 Running Adaptive RAG...

🔁 LOOP 1: what is rag?
  Generated Answer: 'Retrieval Augmented Generation (RAG) combines document retrieval with text generation to reduce hallucinations. Adaptive RAG dynamically rewrites queries, re-evaluates retrieved documents, and validates generated answers.'
  Max similarity to chunks: 0.8432 (Threshold: 0.3)
✅ Answer verified
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3e34aa53dd8d811557.gradio.live


**Use Cases of Adaptive RAG with Hallucination Control (crisp):**

* 📄 **Policy / HR document Q&A** from large PDFs or text files
* 🏥 **Medical guideline lookup** with verified, context-only answers
* ⚖️ **Legal document assistant** to avoid hallucinated clauses
* 🎓 **Study material assistant** for textbooks, notes, research papers
* 🏢 **Company knowledge base chatbot** for internal SOPs & manuals
* 📰 **Research paper exploration** with semantic chunk mapping
* 🛡️ **High-trust domains** where wrong answers are risky
* 🌐 **Fallback to web** when answer not present in document
* 🧠 **Visualization of document meaning** via 3D semantic map
* 🔍 **Hallucination detection system** for safer LLM responses


Core Idea Behind This Project
---

This project demonstrates:

How LLM + Embeddings + Vector DB + Verification can create a trustworthy AI assistant over any document.

✅ Final Conclusion
---

This Adaptive RAG system goes beyond a basic RAG pipeline by introducing:

Semantic understanding of documents

Visual mapping of chunk relationships

Intelligent retrieval

Hallucination prevention

Automatic query correction

Web fallback when document knowledge is insufficient

It showcases how to build a reliable, explainable, and adaptive document QA system using modern LLM and vector search techniques.

---